# From IFC to Explainable Semantic Building Intelligence

## Use case scenario

Imagine receiving an IFC model as part of a BIM handover. The model contains thousands of IFC entities and relationships, but the computer does not automatically understand what they *mean*.

This notebook demonstrates a more intelligent workflow:

> **Can we convert an IFC model into a graph, enrich it with ontology semantics, infer new building knowledge, and explain exactly why each inferred fact is true?**

The scenario is an **explainable BIM handover audit**:

1. Load an IFC file as a `TGraph`.
2. Normalise IFC metadata into TopologicPy ontology keys.
3. Convert the graph into a `KnowledgeGraph`.
4. Run RDFS reasoning over the TopologicPy ontology and BOT mappings.
5. Identify spaces, elements, building containers, and semantic relationships.
6. Prove one inferred fact using a visual proof graph.
7. Write inferred classes back into the `TGraph` dictionaries.

The key point is not only that the system can infer that an IFC object is, for example, a `bot:Space` or `bot:Element`, but that it can show a transparent proof path for that conclusion.

## 0. Import TopologicPy and supporting libraries

Run this notebook after copying the latest replacement files into your local `topologicpy` package and restarting the Jupyter kernel.

If you run this notebook from the root of a local TopologicPy repository, the optional `./src` path is added automatically.

In [1]:
from pathlib import Path
import sys
import inspect
from pprint import pprint
from collections import Counter, defaultdict
import math
import re

sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")


from topologicpy.TGraph import TGraph
from topologicpy.KnowledgeGraph import KnowledgeGraph
from topologicpy.Reasoner import Reasoner
from topologicpy.Ontology import Ontology
from topologicpy.Plotly import Plotly

try:
    import pandas as pd
except Exception:
    pd = None

print("TGraph loaded from:        ", inspect.getfile(TGraph))
print("KnowledgeGraph loaded from:", inspect.getfile(KnowledgeGraph))
print("Reasoner loaded from:      ", inspect.getfile(Reasoner))
print("Ontology loaded from:      ", inspect.getfile(Ontology))
print("Plotly loaded from:        ", inspect.getfile(Plotly))

TGraph loaded from:         C:\Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src\topologicpy\TGraph.py
KnowledgeGraph loaded from: C:\Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src\topologicpy\KnowledgeGraph.py
Reasoner loaded from:       C:\Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src\topologicpy\Reasoner.py
Ontology loaded from:       C:\Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src\topologicpy\Ontology.py
Plotly loaded from:         C:\Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src\topologicpy\Plotly.py


## 1. Provide the IFC file

Place your IFC file path here.

The notebook assumes you will provide an IFC file at the start. Nothing else in the notebook is hard-coded to a particular model.

In [2]:
# ---------------------------------------------------------------------
# USER INPUT
# ---------------------------------------------------------------------
# Replace this with your IFC file path.
# Examples:
# IFC_PATH = Path("models/duplex.ifc")
# IFC_PATH = Path(r"C:/Users/you/Documents/model.ifc")

# IFC_PATH = Path(r"C:\Users\sarwj\OneDrive - Cardiff University\Documents\GitHub\topologicpy\assets\Ifc2x3_Duplex_Architecture.ifc")
# IFC_PATH = Path(r"C:\Users\sarwj\Downloads\Duplex_MEP_20110907.ifc")
IFC_PATH = Path(r"C:\Users\sarwj\Downloads\Ifc4_Revit_MEP.ifc")

# Import configuration
IMPORT_MODE = "topology"       # Recommended: "topology". Other options: "semantic", "triples", "geometry".
DICTIONARY_MODE = "all"        # Options: "basic", "psets", "all", "full".
SILENT = False

# Optional filters. Leave as None for the full graph.
INCLUDE_TYPES = None           # Example: ["IfcProject", "IfcSite", "IfcBuilding", "IfcBuildingStorey", "IfcSpace", "IfcWall", "IfcDoor"]
EXCLUDE_TYPES = None
INCLUDE_RELS = None
EXCLUDE_RELS = None

if not IFC_PATH.exists():
    raise FileNotFoundError(
        f"IFC file not found: {IFC_PATH.resolve()}\n"
        "Edit IFC_PATH in this cell to point to your IFC file, then re-run the notebook."
    )

print("Using IFC file:", IFC_PATH.resolve())

Using IFC file: C:\Users\sarwj\Downloads\Ifc4_Revit_MEP.ifc


## 2. Import the IFC file as a TGraph

`TGraph.ByIFCPath` extracts IFC entities and relationships into a graph structure. In `topology` mode, IFC entities become graph vertices and IFC relationships become graph edges.

This turns the BIM model into something that can be traversed, queried, analysed, reasoned over, and visualised.

In [ ]:
from topologicpy.IFC import IFC

ifc_graph = IFC.TGraphByPath(
        path = str(IFC_PATH),
        importMode = IMPORT_MODE,
    dictionaryMode=DICTIONARY_MODE,
    includeTypes=INCLUDE_TYPES,
    excludeTypes=EXCLUDE_TYPES,
    includeRels=INCLUDE_RELS,
    excludeRels=EXCLUDE_RELS,
    xMin=-10,
    yMin=-10,
    zMin=0,
    xMax=10,
    yMax=10,
    zMax=3,
    silent=SILENT,
    )

assert isinstance(ifc_graph, TGraph), "IFC import did not return a valid TGraph."

print(ifc_graph)
print("Vertex count:", len(TGraph.Vertices(ifc_graph, asTopologic=False, activ=True)))
print("Edge count:  ", len(TGraph.Edges(ifc_graph, asTopologic=False, active=True)))

TGraph(vertices=65465, edges=56787, bidirectional)


TypeError: TGraph.Vertices() got an unexpected keyword argument 'activeOnly'

In [ ]:
verts = TGraph.Vertices(ifc_graph)
mappings = {}
for v in verts:
    ontology_class = v['dictionary'].get('ontology_class', "Not Known")
    mappings[v['dictionary'].get('IFC_type', "Not Known")] = ontology_class

for k in mappings.keys():
    print(k,":", mappings[k])

In [ ]:
edges = TGraph.Edges(ifc_graph)
mappings = {}
for e in edges:
    ontology_class = e['dictionary'].get('ontology_class', "Not Known")
    mappings[e['dictionary'].get('IFC_type', "Not Known")] = ontology_class

for k in mappings.keys():
    print(k,":", mappings[k])

## 3. Normalise IFC metadata into ontology metadata

IFC dictionaries often use keys such as `IFC_type`, `IFC_global_id`, and `IFC_name`.

The ontology/reasoning layer expects canonical keys such as:

- `ifc_class`
- `ifc_guid`
- `label`
- `ontology_class`
- `category`
- `uri`

This cell creates stable semantic URIs and maps IFC classes into TopologicPy ontology classes. For example:

```text
IfcSpace  -> top:Space
IfcWall   -> top:Wall
IfcDoor   -> top:Door
IfcWindow -> top:Window
```

This is where a raw IFC graph starts becoming a semantic building graph.

In [ ]:
def safe_local_name(value, fallback="item"):
    # Returns a URI-safe local name.
    if value is None or value == "":
        value = fallback
    value = str(value).strip()
    value = re.sub(r"[^A-Za-z0-9_\-]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    if not value:
        value = fallback
    if value[0].isdigit():
        value = "id_" + value
    return value

def dget(d, *keys, default=None):
    for key in keys:
        if isinstance(d, dict) and d.get(key, None) not in [None, ""]:
            return d.get(key)
    return default

def records(graph):
    """
    Returns the live TGraph vertex and edge records.

    Important:
    TGraph.Vertices(..., asTopologic=False) and TGraph.Edges(..., asTopologic=False)
    intentionally return copied dictionaries. That is good for read-only use, but it
    means edits made to those returned records do not update the graph.

    For ontology enrichment in this tutorial, we need the live records.
    """
    if isinstance(graph, TGraph):
        vertices = [v for v in graph._vertices if v.get("active", True)]
        edges = [e for e in graph._edges if e.get("active", True)]
        return vertices, edges
    return [], []

def read_records(graph):
    """Returns safe copied records for display/debugging only."""
    return (
        TGraph.Vertices(graph, asTopologic=False, activeOnly=True) or [],
        TGraph.Edges(graph, asTopologic=False, activeOnly=True) or [],
    )

def class_by_ifc(ifc_class):
    """Maps an IFC class to a TopologicPy ontology class using available Ontology APIs."""
    if ifc_class in [None, ""]:
        return None
    for method_name in ["ClassByIFCClass", "TopClassByIFCClass", "OntologyClassByIFCClass"]:
        method = getattr(Ontology, method_name, None)
        if callable(method):
            try:
                return method(ifc_class, defaultValue=None)
            except TypeError:
                try:
                    return method(ifc_class)
                except Exception:
                    pass
            except Exception:
                pass
    try:
        return Ontology.IFC_TO_TOP.get(str(ifc_class), None)
    except Exception:
        return None

def category_by_class(ontology_class):
    """Maps a TopologicPy ontology class to a category using available Ontology APIs."""
    if ontology_class in [None, ""]:
        return None
    for method_name in ["CategoryByClass", "CategoryByOntologyClass"]:
        method = getattr(Ontology, method_name, None)
        if callable(method):
            try:
                return method(ontology_class, defaultValue=None)
            except TypeError:
                try:
                    return method(ontology_class)
                except Exception:
                    pass
            except Exception:
                pass
    try:
        return Ontology.TOP_CATEGORIES.get(str(ontology_class), None)
    except Exception:
        return None

def infer_ifc_class_from_keys(d):
    """
    Attempts to recover an IFC class from common dictionary fields.

    IFC importers sometimes store class names under IFC_type, ifc_type, type,
    class, Entity, entity, or object_type. This helper keeps the tutorial robust
    across slightly different IFC dictionary schemas.
    """
    value = dget(
        d,
        "ifc_class",
        "IFC_type",
        "ifc_type",
        "IfcClass",
        "class",
        "Class",
        "type",
        "Type",
        "Entity",
        "entity",
        "ObjectType",
        "object_type",
    )
    if value not in [None, ""]:
        return str(value)

    # Last fallback: look for any value that looks like an IFC class name.
    for key, item in list(d.items()):
        if isinstance(item, str) and item.startswith("Ifc"):
            return item
    return None

def enrich_ifc_semantics(graph):
    # Adds canonical ontology keys to graph, vertex, and edge dictionaries.
    vertices, edges = records(graph)

    # TGraph's normaliser is still useful, but we do not rely on it exclusively
    # because IFC dictionaries vary between import modes and files.
    try:
        TGraph.NormalizeOntologyDictionaries(
            graph,
            labelKeys=["label", "IFC_name", "Name", "LongName", "name"],
            categoryKeys=["category", "IFC_type", "ifc_type", "type", "ObjectType"],
            ifcClassKeys=["ifc_class", "IFC_type", "ifc_type", "IfcClass", "class", "type"],
            ifcGUIDKeys=["ifc_guid", "IFC_global_id", "GlobalId", "global_id", "guid"],
            includeGraph=True,
            includeVertices=True,
            includeEdges=True,
        )
    except Exception as exc:
        print("Warning: TGraph.NormalizeOntologyDictionaries failed; continuing with notebook enrichment.")
        print("Error:", exc)

    graph._dictionary.setdefault("label", IFC_PATH.stem)
    graph._dictionary.setdefault("ontology_class", "top:KnowledgeGraph")
    graph._dictionary.setdefault("category", "graph")
    graph._dictionary.setdefault("source", str(IFC_PATH))
    graph._dictionary.setdefault("generated_by", "TGraph.ByIFCPath")
    graph._dictionary.setdefault("uri", "inst:" + safe_local_name(IFC_PATH.stem, "IFCModel"))

    for v in vertices:
        d = v.setdefault("dictionary", {})
        idx = v.get("index", 0)

        ifc_class = infer_ifc_class_from_keys(d)
        ifc_guid = dget(d, "ifc_guid", "IFC_global_id", "GlobalId", "global_id", "guid")
        ifc_name = dget(d, "label", "IFC_name", "Name", "LongName", "name", default=f"IFC object {idx}")

        if ifc_class is not None:
            d["ifc_class"] = str(ifc_class)

        if ifc_guid is not None:
            d["ifc_guid"] = str(ifc_guid)

        d.setdefault("label", str(ifc_name))

        if d.get("ontology_class") in [None, ""]:
            ontology_class = class_by_ifc(ifc_class)
            if ontology_class is not None:
                d["ontology_class"] = ontology_class

        if d.get("category") in [None, ""] and d.get("ontology_class") not in [None, ""]:
            category = category_by_class(d["ontology_class"])
            if category is not None:
                d["category"] = category

        subject_id = ifc_guid or dget(d, "IFC_key", "id", "IFC_id", "GlobalId", default=f"vertex_{idx}")
        d["uri"] = "inst:" + safe_local_name(subject_id, f"vertex_{idx}")

    for e in edges:
        d = e.setdefault("dictionary", {})
        idx = e.get("index", 0)

        ifc_rel = dget(
            d,
            "relationship",
            "predicate",
            "ifc_class",
            "IFC_type",
            "ifc_type",
            "type",
            "Type",
            default="IfcRelationship",
        )
        ifc_guid = dget(d, "ifc_guid", "IFC_global_id", "GlobalId", "global_id", "guid")

        d.setdefault("relationship", str(ifc_rel))
        d.setdefault("label", str(ifc_rel))
        d.setdefault("ontology_class", "top:Relationship")
        d.setdefault("category", "semantic_relationship")

        if d.get("ifc_class") in [None, ""] and str(ifc_rel).startswith("Ifc"):
            d["ifc_class"] = str(ifc_rel)

        if ifc_guid is not None:
            d["ifc_guid"] = str(ifc_guid)

        rel_id = ifc_guid or dget(d, "IFC_key", "id", "IFC_id", default=f"edge_{idx}")
        d["uri"] = "inst:" + safe_local_name(rel_id, f"edge_{idx}")

    return graph

ifc_graph = enrich_ifc_semantics(ifc_graph)

vertices, edges = records(ifc_graph)

diagnostics = {
    "live_vertex_count": len(vertices),
    "live_edge_count": len(edges),
    "vertices_with_uri": sum(1 for v in vertices if v.get("dictionary", {}).get("uri") not in [None, ""]),
    "vertices_with_ifc_class": sum(1 for v in vertices if v.get("dictionary", {}).get("ifc_class") not in [None, ""]),
    "vertices_with_ontology_class": sum(1 for v in vertices if v.get("dictionary", {}).get("ontology_class") not in [None, ""]),
    "edges_with_uri": sum(1 for e in edges if e.get("dictionary", {}).get("uri") not in [None, ""]),
    "edges_with_relationship": sum(1 for e in edges if e.get("dictionary", {}).get("relationship") not in [None, ""]),
}

print("Semantic enrichment diagnostics:")
pprint(diagnostics)

if diagnostics["vertices_with_ontology_class"] == 0:
    print("\nWarning: No vertices received ontology_class values.")
    print("Inspect the first few vertex dictionaries below to identify the IFC class key used by your import mode:")
    for v in vertices[:5]:
        pprint(v.get("dictionary", {}))

validation = TGraph.ValidateOntology(
    ifc_graph,
    requireClass=True,
    requireVertexClasses=False,
    requireEdgeClasses=True,
    requireLabels=False,
    checkClassKnown=True,
    checkCategory=False,
)

print("\nOntology validation:")
pprint(validation)

## 4. BIM graph passport

Before reasoning, let us inspect the model as a graph.

This gives a quick semantic passport of the IFC file:

- which IFC classes are present,
- which TopologicPy ontology classes they map to,
- which relationship types dominate the graph.

In [ ]:
vertices, edges = records(ifc_graph)

def counter_from_records(records_, *keys):
    c = Counter()
    for r in records_:
        d = r.get("dictionary", {}) if isinstance(r, dict) else {}
        value = dget(d, *keys, default="Unknown")
        c[str(value)] += 1
    return c

ifc_type_counts = counter_from_records(vertices, "ifc_class", "IFC_type", "type")
ontology_counts = counter_from_records(vertices, "ontology_class")
category_counts = counter_from_records(vertices, "category")
relationship_counts = counter_from_records(edges, "relationship", "predicate", "ifc_class", "IFC_type")

print("Top IFC classes:")
pprint(ifc_type_counts.most_common(20))

print("\nTop ontology classes:")
pprint(ontology_counts.most_common(20))

print("\nTop relationship types:")
pprint(relationship_counts.most_common(20))

if pd:
    display(pd.DataFrame(ifc_type_counts.most_common(20), columns=["IFC class", "Count"]))
    display(pd.DataFrame(ontology_counts.most_common(20), columns=["Ontology class", "Count"]))
    display(pd.DataFrame(relationship_counts.most_common(20), columns=["Relationship", "Count"]))

## 5. Visualise the IFC graph

This visualisation uses the existing `Plotly.DataByTGraph` path. Vertices are colour-coded by `ontology_class`.

For large IFC files, you may want to filter the graph before plotting. The full graph can be dense.

In [ ]:
vertex_groups = sorted([k for k in ontology_counts.keys() if k != "Unknown"])

data = Plotly.DataByTGraph(
    ifc_graph,
    directed=True,
    vertexGroupKey="ontology_class",
    vertexGroups=vertex_groups,
    vertexLabelKey="label",
    showVertexLabel=False,
    vertexSize=6,
    edgeLabelKey="relationship",
    showEdgeLabel=False,
    edgeWidth=1,
    showVertexLegend=False,
    showEdgeLegend=False,
)

fig = Plotly.FigureByData(
    data,
    width=1000,
    height=750,
    xAxis=False,
    yAxis=False,
    zAxis=False,
    backgroundColor="rgba(0,0,0,0)",
)

fig.update_layout(title="IFC model as a semantic TGraph")
fig.show()

## 6. Convert the IFC TGraph into a KnowledgeGraph

The `KnowledgeGraph` layer turns the graph into RDF-like triples. This is the bridge from graph computation to ontology reasoning.

In [ ]:
kg = KnowledgeGraph.ByTGraph(
    ifc_graph,
    includeOntologyAxioms=False,
    includeDictionaries=True,
    includeBOT=True,
    namespacePrefix="inst",
    useRDFLib=True,
)

print(kg)
pprint(kg.Summary())

print("\nFirst 30 triples:")
for triple in kg.Triples()[:30]:
    print(triple)

## 7. Export RDF/Turtle

This creates an interoperable semantic representation of the IFC graph.

You can save this Turtle string as a `.ttl` file, load it into RDF tools, query it with SPARQL, or publish it as part of a linked building data workflow.

In [ ]:
ttl = kg.RDFString(format="turtle", silent=True)
if ttl is None:
    ttl = kg.TurtleString()

print(ttl[:4000])

## 8. Run reasoning

Now we run RDFS inference using the TopologicPy ontology and BOT bridge mappings.

Examples of inferred knowledge:

```text
IfcSpace -> top:Space -> top:Zone -> top:Cell -> top:Topology
top:Space -> bot:Space

IfcWall -> top:Wall -> top:Element -> top:Topology
top:Wall -> bot:Element
```

The graph now knows more than the IFC file explicitly asserted.

In [ ]:
rdf_graph = kg.RDFGraph(silent=False)
print("Asserted RDF graph triples:", len(rdf_graph))

inferred_graph = Reasoner.Infer(
    rdf_graph,
    profile="rdfs",
    includeOntologyAxioms=True,
    includeBOT=True,
    inplace=False,
    silent=False,
)

print("Inferred RDF graph triples:", len(inferred_graph))
print("Added triples:", len(inferred_graph) - len(rdf_graph))

summary = Reasoner.Summary(rdf_graph, inferred_graph)
pprint(summary)

## 9. Semantic handover audit after inference

This asks: after reasoning, how many IFC objects can be understood as BOT spaces, elements, buildings, storeys, or sites?

This is a simple but powerful BIM handover audit: the model is no longer just a file of IFC entities; it becomes a semantically classified building graph.

In [ ]:
def qname(term):
    try:
        return Reasoner.QName(term)
    except Exception:
        return str(term)

def types_for_uri(rdf_graph, uri):
    """
    Returns compact rdf:type values for a subject.

    This helper is intentionally defensive:
    - first tries Reasoner.Types using the supplied QName/URI;
    - then tries the expanded URI;
    - finally scans the graph for an exact compact subject match.
    """
    if uri in [None, ""]:
        return []

    # Normal path.
    types = Reasoner.Types(rdf_graph, uri, compact=True)
    if types:
        return sorted(set(types))

    # Try expanded URI if uri is a QName.
    expanded = None
    try:
        expanded = Reasoner.ExpandQName(uri, defaultValue=None)
    except Exception:
        expanded = None

    if expanded is not None and expanded != uri:
        types = Reasoner.Types(rdf_graph, expanded, compact=True)
        if types:
            return sorted(set(types))

    # Last-resort scan. This avoids empty results if the graph uses a slightly
    # different URIRef/QName representation than expected.
    out = []
    try:
        from rdflib.namespace import RDF
        for s, p, o in rdf_graph.triples((None, RDF.type, None)):
            if qname(s) == uri or str(s) == str(uri) or str(s) == str(expanded):
                out.append(qname(o))
    except Exception:
        pass

    return sorted(set(out))

def inferred_types_for_vertex(v):
    d = v.get("dictionary", {})
    uri = dget(d, "uri")
    return types_for_uri(inferred_graph, uri)

semantic_audit = Counter()
examples_by_type = defaultdict(list)

for v in vertices:
    d = v.get("dictionary", {})
    types = inferred_types_for_vertex(v)

    # Useful diagnostic if there are no inferred types at all.
    if not types and dget(d, "ontology_class") not in [None, ""]:
        types = [dget(d, "ontology_class")]

    for t in types:
        if t.startswith("bot:") or t in ["top:Space", "top:Element", "top:Building", "top:Storey", "top:Site", "top:Zone", "top:Topology"]:
            semantic_audit[t] += 1
            if len(examples_by_type[t]) < 5:
                examples_by_type[t].append({
                    "uri": dget(d, "uri"),
                    "label": dget(d, "label", "IFC_name", default=""),
                    "ifc_class": dget(d, "ifc_class", "IFC_type", default=""),
                    "ontology_class": dget(d, "ontology_class", default=""),
                    "types": types,
                })

print("Semantic audit:")
pprint(semantic_audit.most_common())

if len(semantic_audit) == 0:
    print("\nNo semantic classes were counted. Diagnostics:")
    print("Vertices with ontology_class:", sum(1 for v in vertices if v.get("dictionary", {}).get("ontology_class") not in [None, ""]))
    print("Vertices with uri:", sum(1 for v in vertices if v.get("dictionary", {}).get("uri") not in [None, ""]))
    print("Asserted RDF graph triples:", len(rdf_graph))
    print("Inferred RDF graph triples:", len(inferred_graph))
    print("\nFirst 10 vertex dictionaries:")
    for v in vertices[:10]:
        pprint(v.get("dictionary", {}))
    print("\nFirst 20 inferred rdf:type triples:")
    try:
        from rdflib.namespace import RDF
        shown = 0
        for s, p, o in inferred_graph.triples((None, RDF.type, None)):
            print(qname(s), "rdf:type", qname(o))
            shown += 1
            if shown >= 20:
                break
    except Exception as exc:
        print("Could not inspect rdf:type triples:", exc)

if pd and len(semantic_audit) > 0:
    display(pd.DataFrame(semantic_audit.most_common(), columns=["Inferred semantic class", "Count"]))

print("\nExamples:")
for k, values in examples_by_type.items():
    print("\n", k)
    for item in values:
        print(" ", item)

## 10. Select a meaningful object to explain

We now select one IFC object for proof-based explanation.

Preference order:

1. an `IfcSpace` / `top:Space`,
2. a `top:Room`,
3. a wall, door, window, slab, or other element,
4. any object with an ontology class.

The notebook then asks:

> **Why does the reasoner believe this object belongs to a higher-level building ontology class?**

In [ ]:
preferred_classes = [
    "top:Space",
    "top:Room",
    "top:Wall",
    "top:Door",
    "top:Window",
    "top:Slab",
    "top:Building",
    "top:Storey",
    "top:Site",
    "top:Element",
]

def choose_showcase_vertex(vertices_):
    for cls in preferred_classes:
        for v in vertices_:
            d = v.get("dictionary", {})
            if dget(d, "ontology_class") == cls and dget(d, "uri") not in [None, ""]:
                return v
    for v in vertices_:
        d = v.get("dictionary", {})
        if dget(d, "ontology_class") not in [None, ""] and dget(d, "uri") not in [None, ""]:
            return v
    return None

showcase = choose_showcase_vertex(vertices)
assert showcase is not None, "No suitable IFC object found for explanation."

showcase_d = showcase["dictionary"]
showcase_uri = showcase_d["uri"]
asserted_class = showcase_d.get("ontology_class")

print("Selected object:")
pprint(showcase_d)

print("\nInferred types:")
showcase_types = types_for_uri(inferred_graph, showcase_uri)
for t in showcase_types:
    print(" -", t)

## 11. Choose an inferred target class

The target class depends on the selected IFC object.

For example:

- a `top:Space` should also be a `bot:Space`;
- a `top:Wall` should also be a `bot:Element`;
- a `top:Building` should also be a `bot:Building`;
- a `top:Storey` should also be a `bot:Storey`.

The target must be present in the inferred RDF graph.

In [ ]:
def target_for_class(cls, inferred_types):
    preferences = {
        "top:Room": ["bot:Space", "top:Space", "top:Zone", "top:Topology"],
        "top:Space": ["bot:Space", "top:Zone", "top:Topology"],
        "top:Wall": ["bot:Element", "top:Element", "top:Topology"],
        "top:Door": ["bot:Element", "top:Element", "top:Topology"],
        "top:Window": ["bot:Element", "top:Element", "top:Topology"],
        "top:Slab": ["bot:Element", "top:Element", "top:Topology"],
        "top:Roof": ["bot:Element", "top:Element", "top:Topology"],
        "top:Column": ["bot:Element", "top:Element", "top:Topology"],
        "top:Beam": ["bot:Element", "top:Element", "top:Topology"],
        "top:Element": ["bot:Element", "top:Topology"],
        "top:Building": ["bot:Building", "top:Zone", "top:Topology"],
        "top:Storey": ["bot:Storey", "top:Zone", "top:Topology"],
        "top:Site": ["bot:Site", "top:Zone", "top:Topology"],
    }
    for candidate in preferences.get(cls, []):
        if candidate in inferred_types:
            return candidate
    for candidate in inferred_types:
        if candidate != cls and (candidate.startswith("bot:") or candidate.startswith("top:")):
            return candidate
    return None

target_class = target_for_class(asserted_class, showcase_types)

print("Asserted class:", asserted_class)
print("Target inferred class:", target_class)

assert target_class is not None, "Could not identify a target inferred class for this object."

## 12. Build a proof graph for the inferred fact

This notebook constructs an explicit proof graph for the selected inference.

The proof follows the RDFS rule:

```text
If:
    X rdf:type A
    A rdfs:subClassOf B

Then:
    X rdf:type B
```

When the chain has several superclass steps, the proof graph shows each step.

In [ ]:
def class_parent_map(include_bot=True):
    # Returns child-to-parent class mapping for TopologicPy and BOT bridge classes.
    parent_map = defaultdict(list)

    for child, parents in getattr(Ontology, "TOP_SUPERCLASSES", {}).items():
        for parent in parents:
            parent_map[child].append(parent)

    if include_bot:
        for child, parent in getattr(Ontology, "TOP_TO_BOT", {}).items():
            parent_map[child].append(parent)

    return parent_map

def find_class_path(source_class, target_class, include_bot=True):
    # Breadth-first search over ontology subclass mappings.
    parent_map = class_parent_map(include_bot=include_bot)

    queue = [(source_class, [source_class])]
    seen = {source_class}

    while queue:
        cls, path = queue.pop(0)
        if cls == target_class:
            return path
        for parent in parent_map.get(cls, []):
            if parent not in seen:
                seen.add(parent)
                queue.append((parent, path + [parent]))

    return None

def build_rdfs_type_proof_graph(subject_uri, subject_label, source_class, target_class):
    # Builds proof-graph data for X rdf:type source_class implies X rdf:type target_class.
    path = find_class_path(source_class, target_class, include_bot=True)
    if path is None:
        raise ValueError(f"No subclass path found from {source_class} to {target_class}")

    nodes = []
    edges = []

    def add_node(node_id, label, node_type, level):
        nodes.append({
            "id": node_id,
            "label": label,
            "type": node_type,
            "level": level,
        })

    current_fact_id = "fact_0"
    add_node(
        current_fact_id,
        f"{subject_label}\nrdf:type\n{path[0]}",
        "asserted_fact",
        0,
    )

    for i in range(len(path) - 1):
        child = path[i]
        parent = path[i + 1]

        axiom_id = f"axiom_{i}"
        rule_id = f"rule_rdfs9_{i}"
        next_fact_id = f"fact_{i + 1}"

        add_node(
            axiom_id,
            f"{child}\nrdfs:subClassOf\n{parent}",
            "premise",
            i * 2,
        )

        add_node(
            rule_id,
            "RDFS9\nsubclass type propagation",
            "rule",
            i * 2 + 1,
        )

        add_node(
            next_fact_id,
            f"{subject_label}\nrdf:type\n{parent}",
            "conclusion" if parent == target_class else "inferred_fact",
            i * 2 + 2,
        )

        edges.append({
            "source": current_fact_id,
            "target": rule_id,
            "label": "premise",
            "type": "premise",
        })
        edges.append({
            "source": axiom_id,
            "target": rule_id,
            "label": "premise",
            "type": "premise",
        })
        edges.append({
            "source": rule_id,
            "target": next_fact_id,
            "label": "derives",
            "type": "derived_by",
        })

        current_fact_id = next_fact_id

    return {
        "nodes": nodes,
        "edges": edges,
        "metadata": {
            "subject_uri": subject_uri,
            "subject_label": subject_label,
            "asserted_class": source_class,
            "target_class": target_class,
            "class_path": path,
        },
    }

showcase_label = showcase_d.get("label", showcase_uri)
proof_graph_data = build_rdfs_type_proof_graph(
    showcase_uri,
    showcase_label,
    asserted_class,
    target_class,
)

print("Proof metadata:")
pprint(proof_graph_data["metadata"])

print("\nProof nodes:", len(proof_graph_data["nodes"]))
print("Proof edges:", len(proof_graph_data["edges"]))

## 13. Visualise the proof graph

This is the critical explainability step.

The figure answers:

> **Why did the reasoner classify this IFC object as the target semantic class?**

The proof graph contains:

- asserted fact nodes,
- ontology axiom nodes,
- RDFS rule nodes,
- inferred conclusion nodes.

In [ ]:
proof_fig = Plotly.FigureByProofGraph(
    proofGraphData=proof_graph_data,
    layout="tree",
    title=f"Proof graph: {showcase_label} inferred as {target_class}",
    showNodeLabel=True,
    showEdgeLabel=True,
    showLegend=True,
    width=2400,
    height=1200,
    backgroundColor="white"
    )

proof_fig.show()

## 14. Export the proof graph to HTML

This creates a standalone interactive proof explanation that can be shared with a reviewer, student, client, or BIM coordinator.

In [ ]:
proof_html_path = f"proof_graph_{safe_local_name(showcase_label)}_{safe_local_name(target_class)}.html"

exported_path = Plotly.ProofGraphHTML(
    proofGraphData=proof_graph_data,
    path=proof_html_path,
    layout="tree",
    title=f"Proof graph: {showcase_label} inferred as {target_class}",
    showNodeLabel=True,
    showEdgeLabel=True,
    includePlotlyJS="cdn",
    autoOpen=False,
)

print("Proof graph exported to:", exported_path)

## 15. Write inferred classes back to the IFC TGraph

The inferred RDF types can be written back to the original `TGraph` dictionaries.

This means the graph now carries both:

- the original IFC metadata, and
- the inferred ontology classes.

This is useful before exporting to CSV, RDF, Neo4j, Kuzu, or a graph machine learning dataset.

In [ ]:
Reasoner.ApplyInferences(
    ifc_graph,
    inferred_graph,
    namespacePrefix="inst",
    typeKey="inferred_ontology_classes",
    botTypeKey="inferred_bot_classes",
    overwrite=True,
)

print("Updated selected object dictionary:")
pprint(showcase_d)

print("\nInferred ontology classes:")
pprint(showcase_d.get("inferred_ontology_classes"))

print("\nInferred BOT classes:")
pprint(showcase_d.get("inferred_bot_classes"))

## 16. Semantic query: find all inferred BOT Spaces and Elements

This cell turns reasoning into a useful BIM query:

- Which model objects are recognised as spaces?
- Which model objects are recognised as elements?

These categories may not have been explicitly stored in the IFC file as BOT classes, but they can be inferred through the ontology bridge.

In [ ]:
bot_space_rows = []
bot_element_rows = []

for v in vertices:
    d = v.get("dictionary", {})
    uri = dget(d, "uri")
    if uri in [None, ""]:
        continue

    types = types_for_uri(inferred_graph, uri)

    row = {
        "label": dget(d, "label", default=""),
        "uri": uri,
        "IFC class": dget(d, "ifc_class", "IFC_type", default=""),
        "ontology class": dget(d, "ontology_class", default=""),
    }

    if "bot:Space" in types:
        bot_space_rows.append(row)
    if "bot:Element" in types:
        bot_element_rows.append(row)

print("BOT Spaces:", len(bot_space_rows))
print("BOT Elements:", len(bot_element_rows))

if pd:
    print("\nFirst BOT Spaces")
    display(pd.DataFrame(bot_space_rows[:20]))

    print("\nFirst BOT Elements")
    display(pd.DataFrame(bot_element_rows[:20]))
else:
    print("\nFirst BOT Spaces")
    pprint(bot_space_rows[:20])

    print("\nFirst BOT Elements")
    pprint(bot_element_rows[:20])

## 17. Optional: create a semantic graph view

The `KnowledgeGraph` can also be converted back into a `TGraph`, where RDF resources become vertices and RDF predicates become graph edges.

This is useful when you want to use ordinary graph analytics on semantic data.

In [ ]:
semantic_view = kg.ToTGraph(includeLiterals=False, directed=True)

print(semantic_view)

semantic_data = Plotly.DataByTGraph(
    semantic_view,
    directed=True,
    vertexLabelKey="label",
    showVertexLabel=False,
    edgeLabelKey="label",
    showEdgeLabel=False,
    vertexSize=5,
    edgeWidth=1,
)

semantic_fig = Plotly.FigureByData(
    semantic_data,
    width=1000,
    height=750,
    xAxis=False,
    yAxis=False,
    zAxis=False,
)

semantic_fig.update_layout(title="RDF semantic graph view derived from the IFC model")
semantic_fig.show()

## 18. What this demonstrates

This notebook demonstrates a complete IFC-to-reasoning workflow:

```text
IFC file
  ↓
TGraph
  ↓
Ontology-normalised dictionaries
  ↓
KnowledgeGraph / RDF
  ↓
RDFS inference
  ↓
BOT and TopologicPy semantic classifications
  ↓
Proof graph visualisation
  ↓
Inferred classes written back to TGraph
```

The larger implication is important:

> TopologicPy can act not only as a geometric and topological modelling library, but as an explainable semantic reasoning layer for building graphs.

This opens the door to:

- explainable BIM QA,
- semantic handover checking,
- ontology-aware graph machine learning,
- rule-based compliance workflows,
- linked building data,
- provenance-aware digital twins.